In [1]:
import pandas as pd
import json

# Adjust the path if the main JSON file is named differently inside the cloned folder
file_path = 'leetcode-problems/merged_problems.json' 

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    df = pd.DataFrame(raw_data)
    print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
    
    # Display the column names, data types, and first 3 rows
    display(df.info())
    display(df.head(3))
    
except FileNotFoundError:
    print("File not found. Check the 'leetcode-problems' folder in VS Code to verify the exact name of the JSON data file.")

File not found. Check the 'leetcode-problems' folder in VS Code to verify the exact name of the JSON data file.


In [2]:
import pandas as pd
import json

file_path = 'leetcode-problems/merged_problems.json' 

with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Extract the list of problems directly from the 'questions' key
df = pd.DataFrame(raw_data['questions'])

# Now we can safely deduplicate
duplicates = df.duplicated(subset=['title']).sum()
print(f"Found {duplicates} duplicate problems.")

df_cleaned = df.drop_duplicates(subset=['title'], keep='first')
print(f"Remaining unique problems: {df_cleaned.shape[0]}")

print("\nActual Columns Available for Normalization:")
print(df_cleaned.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'leetcode-problems/merged_problems.json'

In [ ]:
# 1. Check Difficulty Categories
print("Difficulty levels found:")
print(df_cleaned['difficulty'].value_counts(dropna=False))
print("-" * 40)

# 2. Inspect a single problem's Topics, Examples (Test Cases), and Code Snippets
sample_problem = df_cleaned.iloc[0]

print(f"Sample Problem: {sample_problem['title']}")
print(f"Topics format: {type(sample_problem['topics'])} -> {sample_problem['topics']}")
print(f"Examples format: {type(sample_problem['examples'])}")
print(f"Code Snippets format: {type(sample_problem['code_snippets'])}")

# Preview the first example/test case to see its structure
if isinstance(sample_problem['examples'], list) and len(sample_problem['examples']) > 0:
    print("\nFirst Test Case Preview:")
    print(sample_problem['examples'][0])

Difficulty levels found:
difficulty
Medium    1464
Easy       763
Hard       686
Name: count, dtype: int64
----------------------------------------
Sample Problem: Two Sum
Topics format: <class 'list'> -> ['Array', 'Hash Table']
Examples format: <class 'list'>
Code Snippets format: <class 'dict'>

First Test Case Preview:
{'example_num': 1, 'example_text': 'Input: nums = [2,7,11,15], target = 9\nOutput: [0,1]\nExplanation: Because nums[0] + nums[1] == 9, we return [0, 1].', 'images': []}


In [ ]:
import re

def parse_test_cases(examples_list):
    """Extracts distinct input and output values from the raw example text."""
    parsed_cases = []
    if not isinstance(examples_list, list):
        return parsed_cases
        
    for ex in examples_list:
        text = ex.get('example_text', '')
        
        # Regex to capture everything after 'Input: ' up to 'Output: '
        input_match = re.search(r'Input:\s*(.*?)\nOutput:', text, re.IGNORECASE | re.DOTALL)
        # Regex to capture everything after 'Output: ' up to 'Explanation: ' or end of string
        output_match = re.search(r'Output:\s*(.*?)(?:\nExplanation:|$)', text, re.IGNORECASE | re.DOTALL)
        
        if input_match and output_match:
            parsed_cases.append({
                "input": input_match.group(1).strip(),
                "output": output_match.group(1).strip()
            })
    return parsed_cases

# Apply the parsing function
df_cleaned['test_cases'] = df_cleaned['examples'].apply(parse_test_cases)

# Add the missing architecture columns (to be populated later via LLM or feature engineering)
df_cleaned['complexity'] = None
df_cleaned['common_mistakes'] = None

# Rename target columns to match the architecture schema exactly
df_final = df_cleaned.rename(columns={
    'code_snippets': 'reference_solutions'
})

# Filter down to only the columns required by the database schema
final_columns = [
    'problem_id', 'title', 'description', 'topics', 'difficulty', 
    'constraints', 'test_cases', 'reference_solutions', 'complexity', 'common_mistakes'
]
df_final = df_final[final_columns]

print("Schema Normalization Complete. Target DataFrame shape:", df_final.shape)
print("\nParsed Test Case Example (Two Sum):")
print(df_final.iloc[0]['test_cases'])

Schema Normalization Complete. Target DataFrame shape: (2913, 10)

Parsed Test Case Example (Two Sum):
[{'input': 'nums = [2,7,11,15], target = 9', 'output': '[0,1]'}, {'input': 'nums = [3,2,4], target = 6', 'output': '[1,2]'}, {'input': 'nums = [3,3], target = 6', 'output': '[0,1]'}]


In [ ]:
import sqlite3

# Create and connect to a local SQLite database
conn = sqlite3.connect('dsa_database.db')

# SQLite does not natively support Python lists or dictionaries. 
# We must convert the topics, test_cases, and reference_solutions to JSON strings before insertion.
df_sql = df_final.copy()
columns_to_json = ['topics', 'test_cases', 'reference_solutions', 'constraints']

for col in columns_to_json:
    # Use json.dumps to convert objects to strings, handling potential None values
    df_sql[col] = df_sql[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

# Write the dataframe to a SQL table named 'problems'
df_sql.to_sql('problems', conn, if_exists='replace', index=False)
conn.close()

print("Module 1 Complete: 2,913 problems successfully loaded into 'dsa_database.db'.")

Module 1 Complete: 2,913 problems successfully loaded into 'dsa_database.db'.
